# Song Test List
Builds a small list of 20 randomly selected songs with their genres and valid tags for testing.

In [15]:
import json
import csv
import random
from collections import defaultdict

DATA_DIR    = './data/'
CSV_DIR     = DATA_DIR + 'csv/'
SONGS_PATH  = CSV_DIR + 'songs.csv'
TAGS_PATH   = CSV_DIR + 'tags.csv'
GENRE_EMB   = DATA_DIR + 'embeddings/genre_embeddings.json'
TAG_EMB     = DATA_DIR + 'embeddings/tag_embeddings.json'
OUTPUT_PATH = DATA_DIR + 'test_songs.json'

N_SONGS  = 20
MIN_TAGS = 3
SEED     = 42

In [16]:
# Load valid vocab from embeddings
with open(GENRE_EMB) as f:
    valid_genres = set(json.load(f).keys())

with open(TAG_EMB) as f:
    valid_tags = set(json.load(f).keys())

print(f"Valid genres: {len(valid_genres)}")
print(f"Valid tags:   {len(valid_tags)}")

Valid genres: 1305
Valid tags:   985


In [17]:
def read_csv(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f, delimiter=';')
        raw_headers = next(reader)
        headers = [h.strip().strip('"') for h in raw_headers[0].split(',')]
        return [dict(zip(headers, [p.strip().strip('"') for p in row])) for row in reader if len(row) == len(headers)]

song_meta   = {}
song_genres = defaultdict(set)

for row in read_csv(SONGS_PATH):
    sid   = row['spotify_id']
    genre = row['genre_name']
    song_meta[sid] = {'name': row['name'], 'artist': row['artist']}
    if genre in valid_genres:
        song_genres[sid].add(genre)

print(f"Loaded {len(song_meta):,} songs")

Loaded 203,842 songs


In [18]:
song_tags = defaultdict(list)

for row in read_csv(TAGS_PATH):
    sid = row['song_spotify_id']
    tag = row['tag'].lower().strip()
    pop = int(row['popularity']) if row['popularity'].isdigit() else 0
    if tag in valid_tags:
        song_tags[sid].append((tag, pop))

for sid in song_tags:
    seen = {}
    for tag, pop in song_tags[sid]:
        if tag not in seen or pop > seen[tag]:
            seen[tag] = pop
    song_tags[sid] = sorted(seen.items(), key=lambda x: -x[1])

print(f"Songs with valid tags: {sum(1 for sid in song_tags if len(song_tags[sid]) >= MIN_TAGS):,}")

Songs with valid tags: 21,142


In [19]:
candidates = [
    sid for sid in song_meta
    if len(song_genres.get(sid, [])) > 0
    and len(song_tags.get(sid, [])) >= MIN_TAGS
]

random.seed(SEED)
selected = random.sample(candidates, min(N_SONGS, len(candidates)))
print(f"Selected {len(selected)} songs from {len(candidates):,} candidates")

Selected 20 songs from 19,720 candidates


In [20]:
results = []
for sid in selected:
    entry = {
        'spotify_id': sid,
        'name':       song_meta[sid]['name'],
        'artist':     song_meta[sid]['artist'],
        'genres':     sorted(song_genres[sid]),
        'tags':       song_tags[sid][:10],
    }
    results.append(entry)
    print(f"\n{entry['name']} — {entry['artist']}")
    print(f"  genres: {entry['genres']}")
    print(f"  tags:   {entry['tags']}")

with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {OUTPUT_PATH}")


School Boy Crush — Average White Band
  genres: ['classic funk rock', 'funk', 'memphis soul', 'motown', 'quiet storm']
  tags:   [('funk', 100), ('soul', 22), ('rock', 14), ('jecks', 11), ('funky', 9), ('groovy', 9), ('acoustic', 6), ('instrumental', 3), ('fun', 3), ('noir', 3)]

Torn — Natalie Imbruglia
  genres: ['australian pop', 'europop', 'new wave pop', 'pop rock']
  tags:   [('pop', 100), ('rock', 24), ('female', 8), ('alternative', 7), ('nostalgia', 6), ('romantic', 5), ('relax', 4), ('indie', 3), ('memories', 3), ('nostalgic', 2)]

Walk the Shame — Pony the Pirate
  genres: ['norwegian pop', 'norwegian rock']
  tags:   [('scandinavian', 34), ('pop', 34), ('rock', 34)]

Sandstorm — Darude
  genres: ['eurodance']
  tags:   [('trance', 100), ('techno', 68), ('dance', 66), ('electronic', 45), ('electronica', 23), ('club', 6), ('instrumental', 5), ('eurodance', 5), ('electro', 4), ('pop', 4)]

Say Hello 2 Heaven — Temple Of The Dog
  genres: ['grunge']
  tags:   [('rock', 48), ('a